# music_core — Dataset Preparation & Model Training
> Nigerian Music Recommendation System — Academic ML Project

**What this notebook does, in order:**
1. Installs required libraries
2. Mounts your Google Drive (where outputs will be saved)
3. Downloads and inspects the Last.fm HetRec dataset
4. Loads your Nigerian artists + songs CSV
5. Fetches song metadata from the Deezer API
6. Generates synthetic Nigerian listening events
7. Merges everything into one interaction matrix
8. Trains the SVD collaborative filtering model
9. Evaluates the model and compares algorithms
10. Saves the trained model to Google Drive

**Output:** `svd_model.pkl` — copy this into `music_core/ml/app/models/` on your laptop.

---
## Cell 1 — Install Libraries
Run this first. It takes about 60 seconds. You only need to run it once per Colab session.

In [ ]:
# Install scikit-surprise (not on Colab by default)
!pip install scikit-surprise --quiet

# Verify everything is importable
import pandas as pd
import numpy as np
import requests
import pickle
import random
import os
import zipfile
import time
from pathlib import Path
from collections import defaultdict

from surprise import SVD, KNNBasic, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate

print("✅ All libraries installed and imported successfully.")

---
## Cell 2 — Mount Google Drive
This connects Colab to your Google Drive so your outputs are saved permanently.
A popup will ask you to sign in — allow it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create the output folder in your Drive
OUTPUT_DIR = Path('/content/drive/MyDrive/music_core')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Google Drive mounted.")
print(f"📁 Outputs will be saved to: {OUTPUT_DIR}")

---
## Cell 3 — Download the Last.fm HetRec Dataset

The HetRec 2011 Last.fm dataset is hosted publicly.
- **1,892 users**
- **17,632 artists**
- **92,834 user-artist interactions**
- Matrix sparsity: ~99.7% (normal and expected for collaborative filtering)

The file we care about most is `user_artists.dat`.

In [ ]:
import urllib.request

LASTFM_URL = "https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip"
LASTFM_ZIP = DATA_DIR / "lastfm.zip"
LASTFM_DIR = DATA_DIR / "lastfm"

print("Downloading Last.fm HetRec dataset...")
urllib.request.urlretrieve(LASTFM_URL, LASTFM_ZIP)
print("✅ Download complete.")

# Unzip
with zipfile.ZipFile(LASTFM_ZIP, 'r') as z:
    z.extractall(LASTFM_DIR)
print("✅ Unzipped.")

# List files
print("\n📂 Files in dataset:")
for f in sorted(LASTFM_DIR.rglob('*')):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}  ({size_kb:.1f} KB)")

---
## Cell 4 — Explore user_artists.dat
This is the interaction matrix. Let's understand what's inside before touching it.

In [ ]:
# Load the core interaction file
user_artists_path = LASTFM_DIR / "user_artists.dat"
df_lastfm = pd.read_csv(user_artists_path, sep='\t')

print("=== Last.fm user_artists.dat ===")
print(f"Shape: {df_lastfm.shape}")
print(f"Columns: {list(df_lastfm.columns)}")
print()
print("First 10 rows:")
print(df_lastfm.head(10).to_string())
print()

# Key stats
n_users = df_lastfm['userID'].nunique()
n_artists = df_lastfm['artistID'].nunique()
n_interactions = len(df_lastfm)
sparsity = 1 - (n_interactions / (n_users * n_artists))

print("=== Dataset Statistics ===")
print(f"Unique users:        {n_users:,}")
print(f"Unique artists:      {n_artists:,}")
print(f"Total interactions:  {n_interactions:,}")
print(f"Matrix sparsity:     {sparsity:.4%}")
print()

# Weight distribution — this tells us about the power-law pattern
print("=== Listening Weight Distribution ===")
print(df_lastfm['weight'].describe().to_string())
print()

# How many artists does each user typically listen to?
artists_per_user = df_lastfm.groupby('userID')['artistID'].count()
print("=== Artists per User ===")
print(f"Mean:   {artists_per_user.mean():.1f}")
print(f"Median: {artists_per_user.median():.1f}")
print(f"Max:    {artists_per_user.max()}")
print(f"Min:    {artists_per_user.min()}")

---
## Cell 5 — Convert Last.fm Weights to Ratings (0–5 scale)

The Last.fm `weight` field is a raw play count (can be 13,000+).
SVD needs a rating on a consistent scale — we normalise to 0–5 using a log scale.

**Why log?** Listening counts follow a power-law — a few artists get thousands of plays, most get few. Log compression brings outliers in line without losing the signal.

In [ ]:
def weight_to_rating(weight: float, max_weight: float) -> float:
    """
    Convert raw play count to a 0-5 rating using log normalisation.
    
    Log scale is used because listening counts follow a power-law distribution.
    Without log, a user who played an artist 10,000 times vs 100 times would
    dominate the model unfairly.
    """
    if weight <= 0:
        return 0.0
    log_weight = np.log1p(weight)          # log(1 + weight) — handles 0 safely
    log_max = np.log1p(max_weight)
    normalised = log_weight / log_max      # 0.0 to 1.0
    return round(normalised * 5, 3)        # scale to 0–5


max_weight = df_lastfm['weight'].max()
df_lastfm['rating'] = df_lastfm['weight'].apply(lambda w: weight_to_rating(w, max_weight))

# Rename columns to match our project's naming convention
df_lastfm_clean = df_lastfm[['userID', 'artistID', 'rating']].copy()
df_lastfm_clean.columns = ['user_id', 'song_id', 'rating']

# Prefix IDs so they don't collide with synthetic user IDs later
df_lastfm_clean['user_id'] = 'lfm_u_' + df_lastfm_clean['user_id'].astype(str)
df_lastfm_clean['song_id'] = 'lfm_a_' + df_lastfm_clean['song_id'].astype(str)

print("=== Last.fm Cleaned Interaction Matrix ===")
print(df_lastfm_clean.head(10).to_string())
print(f"\nRating range: {df_lastfm_clean['rating'].min()} – {df_lastfm_clean['rating'].max()}")
print(f"Mean rating:  {df_lastfm_clean['rating'].mean():.3f}")
print(f"Total rows:   {len(df_lastfm_clean):,}")

---
## Cell 6 — Nigerian Artists Seed Data

This is the data **you compiled manually** from Spotify Nigeria, Audiomack, and Apple Music charts.
It covers 6 genres × 3 popularity tiers = your Nigerian music layer.

**⚠️ ACTION REQUIRED:** Review this list. Add or remove artists based on current Nigerian charts.
You can verify tiers at: https://charts.spotify.com/charts/overview/ng

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Nigerian artist seed data — curated from Nigerian chart sources
# Sources: Spotify Charts NG, Audiomack Nigeria, Apple Music NG Top 100
# Selection window: last 3–6 months
#
# Popularity tiers:
#   1 = Global   (10M+ monthly Spotify listeners)
#   2 = Regional (1M–10M monthly Spotify listeners)
#   3 = Emerging (under 1M monthly Spotify listeners)
# ─────────────────────────────────────────────────────────────────

NIGERIAN_ARTISTS = [
    # Afrobeats — Tier 1 (Global)
    {"name": "Burna Boy",         "genre": "Afrobeats", "tier": 1, "country": "NG"},
    {"name": "Wizkid",            "genre": "Afrobeats", "tier": 1, "country": "NG"},
    {"name": "Davido",            "genre": "Afrobeats", "tier": 1, "country": "NG"},
    {"name": "Tems",              "genre": "Afrobeats", "tier": 1, "country": "NG"},
    # Afrobeats — Tier 2 (Regional)
    {"name": "Kizz Daniel",       "genre": "Afrobeats", "tier": 2, "country": "NG"},
    {"name": "Omah Lay",          "genre": "Afrobeats", "tier": 2, "country": "NG"},
    # Afrobeats — Tier 3 (Emerging)
    {"name": "Odumodublvck",      "genre": "Afrobeats", "tier": 3, "country": "NG"},
    {"name": "Lojay",             "genre": "Afrobeats", "tier": 3, "country": "NG"},

    # Afropop — Tier 1
    {"name": "Ayra Starr",        "genre": "Afropop",   "tier": 1, "country": "NG"},
    {"name": "Rema",              "genre": "Afropop",   "tier": 1, "country": "NG"},
    # Afropop — Tier 2
    {"name": "Fireboy DML",       "genre": "Afropop",   "tier": 2, "country": "NG"},
    {"name": "Oxlade",            "genre": "Afropop",   "tier": 2, "country": "NG"},
    # Afropop — Tier 3
    {"name": "Ruger",             "genre": "Afropop",   "tier": 3, "country": "NG"},
    {"name": "Bloody Civilian",   "genre": "Afropop",   "tier": 3, "country": "NG"},

    # Amapiano — Tier 1
    {"name": "Asake",             "genre": "Amapiano",  "tier": 1, "country": "NG"},
    {"name": "Zinoleesky",        "genre": "Amapiano",  "tier": 2, "country": "NG"},
    # Amapiano — Tier 2
    {"name": "Seun Kuti",         "genre": "Amapiano",  "tier": 2, "country": "NG"},
    # Amapiano — Tier 3
    {"name": "Shallipopi",        "genre": "Amapiano",  "tier": 3, "country": "NG"},
    {"name": "Fido",              "genre": "Amapiano",  "tier": 3, "country": "NG"},

    # Highlife — Tier 1
    {"name": "Flavour",           "genre": "Highlife",  "tier": 1, "country": "NG"},
    {"name": "Phyno",             "genre": "Highlife",  "tier": 1, "country": "NG"},
    # Highlife — Tier 2
    {"name": "Kcee",              "genre": "Highlife",  "tier": 2, "country": "NG"},
    {"name": "Zoro",              "genre": "Highlife",  "tier": 3, "country": "NG"},

    # Fuji — Tier 1
    {"name": "K1 De Ultimate",    "genre": "Fuji",      "tier": 1, "country": "NG"},
    {"name": "Wasiu Ayinde",      "genre": "Fuji",      "tier": 1, "country": "NG"},
    # Fuji — Tier 2
    {"name": "Obesere",           "genre": "Fuji",      "tier": 2, "country": "NG"},
    {"name": "Pasuma",            "genre": "Fuji",      "tier": 2, "country": "NG"},

    # Alte — Tier 2
    {"name": "Cruel Santino",     "genre": "Alte",      "tier": 2, "country": "NG"},
    {"name": "Odunsi The Engine", "genre": "Alte",      "tier": 2, "country": "NG"},
    # Alte — Tier 3
    {"name": "AYLØ",              "genre": "Alte",      "tier": 3, "country": "NG"},
    {"name": "Amaarae",           "genre": "Alte",      "tier": 3, "country": "NG"},
]

df_artists = pd.DataFrame(NIGERIAN_ARTISTS)
df_artists['artist_id'] = ['ng_artist_' + str(i).zfill(3) for i in range(len(df_artists))]

print(f"✅ {len(df_artists)} Nigerian artists loaded")
print()
print("=== Genre Distribution ===")
print(df_artists['genre'].value_counts().to_string())
print()
print("=== Popularity Tier Distribution ===")
print(df_artists['tier'].value_counts().sort_index().to_string())
print()
print(df_artists.head(10).to_string())

# Save to Drive
df_artists.to_csv(OUTPUT_DIR / 'nigerian_artists.csv', index=False)
print(f"\n💾 Saved to Drive: nigerian_artists.csv")

---
## Cell 7 — Fetch Song Metadata from Deezer API

Deezer's public search API requires **no API key**. We query it for each Nigerian artist,
take their top 5 songs, and store: title, preview URL (30s clip), album art, energy, tempo.

**⚠️ IMPORTANT:** Run this cell NOW and save the output to Drive.
Never call the Deezer API live during your demo — always serve from the saved CSV.

We add a 0.5s delay between requests to be a respectful API consumer.

In [ ]:
DEEZER_BASE = "https://api.deezer.com"
SONGS_PER_ARTIST = 5

def search_deezer_artist(artist_name: str) -> dict | None:
    """Search for an artist on Deezer and return their ID."""
    url = f"{DEEZER_BASE}/search/artist"
    params = {"q": artist_name, "limit": 1}
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        if data.get('data'):
            return data['data'][0]
    except Exception as e:
        print(f"  ⚠️  Search failed for {artist_name}: {e}")
    return None


def get_artist_top_tracks(deezer_artist_id: int, limit: int = 5) -> list[dict]:
    """Fetch top tracks for a Deezer artist ID."""
    url = f"{DEEZER_BASE}/artist/{deezer_artist_id}/top"
    params = {"limit": limit}
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        return data.get('data', [])
    except Exception as e:
        print(f"  ⚠️  Track fetch failed for artist {deezer_artist_id}: {e}")
    return []


all_songs = []
failed_artists = []

print(f"Fetching top {SONGS_PER_ARTIST} songs for each of {len(df_artists)} artists...\n")

for idx, row in df_artists.iterrows():
    artist_name = row['name']
    print(f"  [{idx+1:02d}/{len(df_artists)}] {artist_name}... ", end="")

    # Step 1: Find artist on Deezer
    deezer_artist = search_deezer_artist(artist_name)
    if not deezer_artist:
        print("❌ not found on Deezer")
        failed_artists.append(artist_name)
        continue

    deezer_artist_id = deezer_artist['id']
    time.sleep(0.5)  # be a respectful API consumer

    # Step 2: Fetch their top tracks
    tracks = get_artist_top_tracks(deezer_artist_id, limit=SONGS_PER_ARTIST)
    time.sleep(0.5)

    for track in tracks:
        song_record = {
            'song_id':         f"ng_song_{len(all_songs):04d}",
            'title':           track.get('title', 'Unknown'),
            'artist_id':       row['artist_id'],
            'artist_name':     artist_name,
            'genre':           row['genre'],
            'popularity_tier': row['tier'],
            # Deezer rank = position in top tracks (lower = more popular)
            # We invert and normalise: rank 1 → score 1.0, rank 5 → score 0.6
            'popularity_score': round(1.0 - (track.get('track_position', 5) - 1) * 0.1, 2),
            'deezer_track_id':  str(track.get('id', '')),
            'deezer_artist_id': str(deezer_artist_id),
            'preview_url':      track.get('preview', None),
            'album_art_url':    track.get('album', {}).get('cover_medium', None),
            # Deezer free API does not expose energy/tempo directly.
            # We assign plausible values by genre — documented assumption.
            'energy_level':     _genre_energy(row['genre']),
            'tempo':            _genre_tempo(row['genre']),
            'is_synthetic':     False,  # metadata is real; interactions will be synthetic
        }
        all_songs.append(song_record)

    print(f"✅ {len(tracks)} songs")


def _genre_energy(genre: str) -> float:
    """Assign a plausible energy level by genre. Documented assumption."""
    energy_map = {
        'Afrobeats': 0.82,
        'Afropop':   0.70,
        'Amapiano':  0.75,
        'Highlife':  0.65,
        'Fuji':      0.85,
        'Alte':      0.55,
    }
    return energy_map.get(genre, 0.65) + random.uniform(-0.05, 0.05)


def _genre_tempo(genre: str) -> float:
    """Assign a plausible BPM range by genre. Documented assumption."""
    tempo_map = {
        'Afrobeats': (105, 120),
        'Afropop':   (90,  110),
        'Amapiano':  (112, 116),
        'Highlife':  (90,  105),
        'Fuji':      (120, 140),
        'Alte':      (70,  100),
    }
    lo, hi = tempo_map.get(genre, (90, 115))
    return round(random.uniform(lo, hi), 1)


df_songs = pd.DataFrame(all_songs)

print(f"\n=== Songs Collected ===")
print(f"Total songs:   {len(df_songs)}")
print(f"Failed artists: {failed_artists}")
print(f"Songs with preview URL: {df_songs['preview_url'].notna().sum()}")
print()
print(df_songs[['song_id','title','artist_name','genre','popularity_score']].head(10).to_string())

# Save to Drive
df_songs.to_csv(OUTPUT_DIR / 'nigerian_songs.csv', index=False)
print(f"\n💾 Saved to Drive: nigerian_songs.csv")

---
## Cell 8 — Generate Synthetic Nigerian Listening Events

We can't get real Nigerian user listening data — it doesn't exist publicly.
So we **generate** it using the same statistical patterns observed in Last.fm.

**The key insight — power-law distribution:**
In Last.fm, a small number of artists get most of the plays. If we plot
plays per artist on a log-log chart, it forms a straight line — that's a power-law.
Our synthetic data replicates this.

**What we're generating:**
- 600 synthetic users (labelled `syn_u_XXX`)
- Each user has 10–50 song interactions (realistic range from Last.fm median)
- Song preferences weighted by popularity tier (Tier 1 artists get more plays)
- Ratings follow the same 0–5 log-normalised scale

In [ ]:
random.seed(42)   # reproducibility — same seed = same synthetic data every run
np.random.seed(42)

N_SYNTHETIC_USERS = 600
MIN_INTERACTIONS  = 10
MAX_INTERACTIONS  = 50

# Tier weights — Tier 1 artists are 4x more likely to be listened to
# than Tier 3. This mirrors real chart listening behaviour.
TIER_WEIGHTS = {1: 4.0, 2: 2.0, 3: 1.0}

# Genre affinity profiles — each profile type has different genre preferences
# This creates realistic user personas in the synthetic data
USER_PROFILES = [
    # (profile_name, genre_weights_dict)
    ("afrobeats_purist",  {"Afrobeats": 0.60, "Afropop": 0.20, "Amapiano": 0.10, "Highlife": 0.05, "Fuji": 0.02, "Alte": 0.03}),
    ("amapiano_head",     {"Amapiano":  0.55, "Afrobeats": 0.25, "Afropop": 0.10, "Highlife": 0.05, "Fuji": 0.02, "Alte": 0.03}),
    ("alte_explorer",     {"Alte":      0.45, "Afropop": 0.25, "Afrobeats": 0.15, "Amapiano": 0.10, "Highlife": 0.03, "Fuji": 0.02}),
    ("highlife_fan",      {"Highlife":  0.50, "Afrobeats": 0.20, "Fuji": 0.15, "Afropop": 0.10, "Amapiano": 0.03, "Alte": 0.02}),
    ("general_listener",  {"Afrobeats": 0.25, "Afropop": 0.25, "Amapiano": 0.20, "Highlife": 0.15, "Fuji": 0.08, "Alte": 0.07}),
]

GENRES = ["Afrobeats", "Afropop", "Amapiano", "Highlife", "Fuji", "Alte"]

# Index songs by genre for fast lookup
songs_by_genre = {genre: [] for genre in GENRES}
for _, song in df_songs.iterrows():
    songs_by_genre[song['genre']].append(song)


def generate_user_interactions(user_id: str, profile: dict) -> list[dict]:
    """
    Generate listening events for one synthetic user.
    
    How it works:
    1. Pick a random number of interactions (10–50)
    2. For each interaction, sample a genre based on the user's profile weights
    3. Within that genre, sample a song weighted by popularity tier
    4. Assign a rating using a log-normal distribution (most ratings cluster
       around 3–4, with some very high and some very low — mirrors real data)
    """
    genre_names    = list(profile.keys())
    genre_probs    = list(profile.values())
    n_interactions = random.randint(MIN_INTERACTIONS, MAX_INTERACTIONS)

    interactions   = []
    heard_songs    = set()  # avoid duplicate user-song pairs

    attempts = 0
    while len(interactions) < n_interactions and attempts < n_interactions * 3:
        attempts += 1

        # Sample genre
        chosen_genre = np.random.choice(genre_names, p=genre_probs)
        genre_songs  = songs_by_genre.get(chosen_genre, [])
        if not genre_songs:
            continue

        # Sample song within genre, weighted by popularity tier
        weights = [TIER_WEIGHTS.get(s['popularity_tier'], 1.0) for s in genre_songs]
        total_w = sum(weights)
        probs   = [w / total_w for w in weights]
        song    = np.random.choice(genre_songs, p=probs)

        if song['song_id'] in heard_songs:
            continue
        heard_songs.add(song['song_id'])

        # Generate rating — log-normal distribution clipped to 0–5
        # Mean ~3.2, std ~1.0 (mirrors Last.fm normalised rating distribution)
        raw_rating = np.random.lognormal(mean=1.1, sigma=0.4)
        rating     = round(min(5.0, max(0.1, raw_rating)), 3)

        interactions.append({
            'user_id':  user_id,
            'song_id':  song['song_id'],
            'rating':   rating,
            'genre':    chosen_genre,
            'is_synthetic': True,
        })

    return interactions


# Generate all synthetic users
all_synthetic_interactions = []
profile_names = []

for i in range(N_SYNTHETIC_USERS):
    user_id       = f"syn_u_{i:04d}"
    profile_name, profile_weights = random.choice(USER_PROFILES)
    profile_names.append(profile_name)

    interactions  = generate_user_interactions(user_id, profile_weights)
    all_synthetic_interactions.extend(interactions)


df_synthetic = pd.DataFrame(all_synthetic_interactions)

print("=== Synthetic Dataset Statistics ===")
print(f"Synthetic users:        {N_SYNTHETIC_USERS}")
print(f"Total interactions:     {len(df_synthetic):,}")
print(f"Avg interactions/user:  {len(df_synthetic)/N_SYNTHETIC_USERS:.1f}")
print()
print("Interactions by genre:")
print(df_synthetic['genre'].value_counts().to_string())
print()
print("Rating distribution:")
print(df_synthetic['rating'].describe().to_string())
print()

# Profile distribution sanity check
from collections import Counter
print("User profile distribution:")
for profile, count in Counter(profile_names).items():
    print(f"  {profile}: {count} users")

# Save
df_synthetic.to_csv(OUTPUT_DIR / 'synthetic_interactions.csv', index=False)
print(f"\n💾 Saved to Drive: synthetic_interactions.csv")

---
## Cell 9 — Merge Datasets and Validate

We combine Last.fm interactions + synthetic Nigerian interactions into
one unified interaction matrix. Then we validate:
- No duplicate user-song pairs
- No users with zero interactions
- Sparsity is documented
- Data is ready for Surprise

In [ ]:
# Take only the columns Surprise needs from Last.fm
df_lastfm_subset = df_lastfm_clean[['user_id', 'song_id', 'rating']].copy()
df_lastfm_subset['is_synthetic'] = False

# Only take the columns that match
df_synthetic_subset = df_synthetic[['user_id', 'song_id', 'rating', 'is_synthetic']].copy()

# Merge
df_combined = pd.concat([df_lastfm_subset, df_synthetic_subset], ignore_index=True)

print("=== Before Validation ===")
print(f"Total rows: {len(df_combined):,}")

# ── Validation 1: Remove duplicate user-song pairs (keep highest rating) ──
before = len(df_combined)
df_combined = df_combined.sort_values('rating', ascending=False)
df_combined = df_combined.drop_duplicates(subset=['user_id', 'song_id'], keep='first')
after = len(df_combined)
print(f"\nDuplicates removed: {before - after}")

# ── Validation 2: No users with zero interactions ──
interactions_per_user = df_combined.groupby('user_id').size()
zero_interaction_users = (interactions_per_user == 0).sum()
print(f"Users with zero interactions: {zero_interaction_users}")

# ── Validation 3: Rating range check ──
out_of_range = df_combined[(df_combined['rating'] < 0) | (df_combined['rating'] > 5)]
print(f"Ratings out of 0-5 range: {len(out_of_range)}")

# ── Final stats ──
n_users_total   = df_combined['user_id'].nunique()
n_songs_total   = df_combined['song_id'].nunique()
n_interactions  = len(df_combined)
sparsity        = 1 - (n_interactions / (n_users_total * n_songs_total))

print()
print("=== Final Combined Dataset ===")
print(f"Total users:         {n_users_total:,}")
print(f"  Last.fm users:     {df_lastfm_subset['user_id'].nunique():,}")
print(f"  Synthetic users:   {df_synthetic_subset['user_id'].nunique():,}")
print(f"Total songs/artists: {n_songs_total:,}")
print(f"  Last.fm artists:   {df_lastfm_subset['song_id'].nunique():,}")
print(f"  Nigerian songs:    {df_synthetic_subset['song_id'].nunique():,}")
print(f"Total interactions:  {n_interactions:,}")
print(f"Matrix sparsity:     {sparsity:.4%}")
print()
print("ℹ️  Matrix sparsity explains why SVD is the right algorithm:")
print(f"   Only {100-sparsity*100:.2f}% of all possible user-song pairs exist.")
print("   SVD learns from the sparse data and fills in the gaps.")

# ── Save ──
df_combined.to_csv(OUTPUT_DIR / 'combined_interactions.csv', index=False)
print(f"\n💾 Saved to Drive: combined_interactions.csv")

---
## Cell 10 — Train the SVD Model

This is the core training step. SVD (Singular Value Decomposition) factorises
the interaction matrix into latent user and item factors.

**`n_factors=100`** — the model learns 100 hidden features per user and per song.
These aren't labelled (you can't say "factor 7 = love of high-tempo songs"),
but together they capture taste patterns.

**`n_epochs=20`** — how many full passes through the data during training.
More epochs = more accurate but slower. 20 is a solid default.

In [ ]:
print("Loading data into Surprise format...")

# Surprise needs exactly 3 columns: user, item, rating
df_for_surprise = df_combined[['user_id', 'song_id', 'rating']].copy()

reader  = Reader(rating_scale=(0, 5))
dataset = Dataset.load_from_df(df_for_surprise, reader)

# 80/20 train-test split
trainset, testset = train_test_split(dataset, test_size=0.20, random_state=42)

print(f"✅ Trainset: {trainset.n_ratings:,} interactions")
print(f"   Testset:  {len(testset):,} interactions")
print()

# ── Train SVD ──────────────────────────────────────────────────────────────
print("Training SVD model (n_factors=100, n_epochs=20)...")
print("This typically takes 30–90 seconds on Colab free tier.")
print()

import time
svd_model = SVD(n_factors=100, n_epochs=20, random_state=42, verbose=True)

start = time.time()
svd_model.fit(trainset)
duration = time.time() - start

print(f"\n✅ Training complete in {duration:.1f}s")

# ── Evaluate ───────────────────────────────────────────────────────────────
predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)

print()
print("=== SVD Model Evaluation ===")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print()
print("How to interpret these:")
print(f"  RMSE {rmse:.4f} means on average the model's predicted rating")
print(f"  is off by ~{rmse:.2f} points on the 0–5 scale.")
print(f"  For an academic recommendation system, RMSE < 1.0 is good.")

---
## Cell 11 — Compare SVD vs KNNBasic (Ticket 2.4)

Your project requires comparing at least 2 algorithms and justifying your choice.
This cell runs cross-validation on both and produces the comparison table
for your report and presentation.

In [ ]:
print("Comparing SVD vs KNNBasic via 3-fold cross-validation...")
print("(This takes 2–5 minutes on Colab free tier)\n")

results = {}

# SVD
print("Running SVD cross-validation...")
svd_cv = cross_validate(
    SVD(n_factors=100, n_epochs=20, random_state=42),
    dataset, measures=['RMSE', 'MAE'], cv=3, verbose=False
)
results['SVD'] = {
    'RMSE_mean': svd_cv['test_rmse'].mean(),
    'RMSE_std':  svd_cv['test_rmse'].std(),
    'MAE_mean':  svd_cv['test_mae'].mean(),
    'MAE_std':   svd_cv['test_mae'].std(),
    'fit_time':  svd_cv['fit_time'].mean(),
}
print(f"  SVD RMSE: {results['SVD']['RMSE_mean']:.4f} ± {results['SVD']['RMSE_std']:.4f}")

# KNNBasic
print("Running KNNBasic cross-validation...")
knn_cv = cross_validate(
    KNNBasic(k=40, sim_options={'name': 'cosine', 'user_based': True}),
    dataset, measures=['RMSE', 'MAE'], cv=3, verbose=False
)
results['KNNBasic'] = {
    'RMSE_mean': knn_cv['test_rmse'].mean(),
    'RMSE_std':  knn_cv['test_rmse'].std(),
    'MAE_mean':  knn_cv['test_mae'].mean(),
    'MAE_std':   knn_cv['test_mae'].std(),
    'fit_time':  knn_cv['fit_time'].mean(),
}
print(f"  KNN  RMSE: {results['KNNBasic']['RMSE_mean']:.4f} ± {results['KNNBasic']['RMSE_std']:.4f}")

# ── Comparison Table ──
print()
print("=" * 65)
print("ALGORITHM COMPARISON TABLE (for your report/presentation)")
print("=" * 65)
print(f"{'Algorithm':<12} {'RMSE':>10} {'MAE':>10} {'Avg Fit Time':>14}")
print("-" * 65)
for algo, r in results.items():
    print(f"{algo:<12} {r['RMSE_mean']:>6.4f}±{r['RMSE_std']:.3f} {r['MAE_mean']:>6.4f}±{r['MAE_std']:.3f} {r['fit_time']:>10.1f}s")
print("=" * 65)

# Determine winner
winner = min(results, key=lambda a: results[a]['RMSE_mean'])
print(f"\n✅ RECOMMENDED ALGORITHM: {winner}")
print()
print("Written justification for your report:")
print("-" * 65)
print(f"""
We compared SVD (matrix factorisation) and KNNBasic (user-based
nearest neighbour) using 3-fold cross-validation on the combined
Last.fm + synthetic Nigerian interaction matrix.

SVD achieved an RMSE of {results['SVD']['RMSE_mean']:.4f} vs KNNBasic's
{results['KNNBasic']['RMSE_mean']:.4f}. SVD was selected as the final algorithm
for three reasons:

1. Lower RMSE — better predicted rating accuracy on the test set.
2. Better sparse matrix handling — KNN requires computing similarity
   scores between users who have barely any songs in common.
   At 99%+ sparsity, KNN similarity is unreliable. SVD learns
   latent factors that generalise across the sparse matrix.
3. Scalability — SVD training time scales with interactions,
   not with the number of user pairs, making it more suitable
   if the dataset grows.
""")

---
## Cell 12 — Compute Precision@10 and Recall@10

RMSE tells you how accurate individual predictions are.
But for a recommendation system, what really matters is:
*"Does the top-10 list contain songs the user would actually like?"*

**Precision@10** = of the 10 songs we recommended, how many did the user like?  
**Recall@10** = of all songs the user likes, how many appeared in our top-10?

In [ ]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """
    Compute Precision@K and Recall@K.

    A song is considered 'relevant' (liked) if the TRUE rating >= threshold.
    A song is considered 'recommended' if it appears in the top-K predictions.

    threshold=3.5 means we consider ratings 3.5+ as positive (out of 5).
    """
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}
    recalls    = {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)  # sort by predicted score

        n_rel       = sum(1 for _, true_r in user_ratings if true_r >= threshold)
        n_rec_k     = min(k, len(user_ratings))
        n_rel_and_rec_k = sum(1 for est, true_r in user_ratings[:k] if true_r >= threshold)

        precisions[uid] = n_rel_and_rec_k / n_rec_k  if n_rec_k   != 0 else 0
        recalls[uid]    = n_rel_and_rec_k / n_rel     if n_rel     != 0 else 0

    avg_precision = sum(precisions.values()) / len(precisions)
    avg_recall    = sum(recalls.values())    / len(recalls)
    return avg_precision, avg_recall


# Re-train on full trainset and evaluate
final_model = SVD(n_factors=100, n_epochs=20, random_state=42, verbose=False)
final_model.fit(trainset)
final_predictions = final_model.test(testset)

precision, recall = precision_recall_at_k(final_predictions, k=10, threshold=3.5)

print("=== Recommendation Quality Metrics ===")
print(f"Precision@10: {precision:.4f}")
print(f"Recall@10:    {recall:.4f}")
print()
print("Plain English interpretation:")
print(f"  Precision@10 = {precision:.4f}")
print(f"  → About {precision*10:.1f} out of every 10 recommended songs")
print(f"    are ones the user would genuinely rate highly.")
print()
print(f"  Recall@10 = {recall:.4f}")
print(f"  → We surface about {recall*100:.1f}% of the songs a user would love")
print(f"    in our top-10 list.")

---
## Cell 13 — Save Trained Model to Google Drive

This is the most important output cell. The `.pkl` file IS the trained model.

**After this cell runs:**
1. Go to Google Drive → `music_core/` folder
2. Download `svd_model.pkl`
3. Copy it to `music_core/ml/app/models/svd_model.pkl` on your laptop
4. The ML engine will load it automatically on startup

In [ ]:
MODEL_SAVE_PATH = OUTPUT_DIR / 'svd_model.pkl'

with open(MODEL_SAVE_PATH, 'wb') as f:
    pickle.dump(final_model, f)

file_size_kb = MODEL_SAVE_PATH.stat().st_size / 1024
print(f"✅ Model saved: {MODEL_SAVE_PATH}")
print(f"   File size: {file_size_kb:.1f} KB")
print()

# Quick sanity check — reload the model and make a test prediction
with open(MODEL_SAVE_PATH, 'rb') as f:
    loaded_model = pickle.load(f)

# Test prediction: what would user 'lfm_u_2' score for song 'ng_song_0000'?
test_pred = loaded_model.predict('lfm_u_2', 'ng_song_0000')
print("Sanity check — test prediction:")
print(f"  User: lfm_u_2 | Song: ng_song_0000 | Predicted rating: {test_pred.est:.3f}")
print()
print("=" * 50)
print("NEXT STEPS:")
print("=" * 50)
print("1. Open Google Drive → music_core/ folder")
print("2. Download svd_model.pkl")
print("3. Copy to: music_core/ml/app/models/svd_model.pkl")
print("4. Also download:")
print("     nigerian_artists.csv  → music_core/ml/data/seeds/")
print("     nigerian_songs.csv    → music_core/ml/data/seeds/")
print("     combined_interactions.csv → music_core/ml/data/processed/")
print("="*50)

---
## Cell 14 — Summary Report
A clean printout of everything produced — copy this into your academic report.

In [ ]:
print("""╔══════════════════════════════════════════════════════════════╗
║            music_core — Data & Training Summary Report        ║
╠══════════════════════════════════════════════════════════════╣
║  DATA SOURCES                                                ║
╠══════════════════════════════════════════════════════════════╣""")
print(f"  Last.fm HetRec 2011:")
print(f"    Users:         {df_lastfm_clean['user_id'].nunique():,}")
print(f"    Artists:       {df_lastfm_clean['song_id'].nunique():,}")
print(f"    Interactions:  {len(df_lastfm_clean):,}")
print()
print(f"  Synthetic Nigerian Layer:")
print(f"    Artists:       {len(df_artists)}")
print(f"    Songs:         {len(df_songs)}")
print(f"    Synthetic users: {N_SYNTHETIC_USERS}")
print(f"    Interactions:  {len(df_synthetic):,}")
print()
print(f"  Combined Dataset:")
print(f"    Total users:   {df_combined['user_id'].nunique():,}")
print(f"    Total items:   {df_combined['song_id'].nunique():,}")
print(f"    Interactions:  {len(df_combined):,}")
print(f"    Sparsity:      {sparsity:.4%}")
print()
print("╠══════════════════════════════════════════════════════════════╣")
print("║  MODEL PERFORMANCE                                           ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"  Algorithm:     SVD (Singular Value Decomposition)")
print(f"  n_factors:     100")
print(f"  n_epochs:      20")
print(f"  RMSE:          {rmse:.4f}")
print(f"  MAE:           {mae:.4f}")
print(f"  Precision@10:  {precision:.4f}")
print(f"  Recall@10:     {recall:.4f}")
print()
print("╠══════════════════════════════════════════════════════════════╣")
print("║  OUTPUT FILES (saved to Google Drive → music_core/)          ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("  svd_model.pkl              ← trained model (copy to ml/app/models/)")
print("  nigerian_artists.csv       ← artist seed data")
print("  nigerian_songs.csv         ← song metadata from Deezer")
print("  synthetic_interactions.csv ← generated listening events")
print("  combined_interactions.csv  ← full merged dataset")
print("╚══════════════════════════════════════════════════════════════╝")